# Classifying by One Variable

Data scientists often need to classify individuals into groups according to shared features, and then identify some characteristics of the groups. For example, in the example using Galton's data on heights, we saw that it was useful to classify families according to the parents' midparent heights, and then find the average height of the children in each group.

This section is about classifying individuals into categories that are not numerical. We begin by recalling the basic use of `group`. 

In [ ]:
import polars as pl
import hvplot.polars
path_data = '../../../assets/data/'


## Counting the Number in Each Category
The `group` method with a single argument counts the number of rows for each category in a column. The result contains one row per unique value in the grouped column.

Here is a small DataFrame of data on ice cream cones. The `group` method can be used to list the distinct flavors and provide the counts of each flavor.

In [ ]:
cones = pl.DataFrame([
    pl.Series("Flavor", ["strawberry", "chocolate", "chocolate", "strawberry", "chocolate"]),
    pl.Series("Price", [3.55, 4.75, 6.55, 5.25, 5.25]),
])
cones


In [ ]:
# POLARS_CHANGE: the count column is named "count" because Polars would call it "len". group_by does not sort, so the categories are sorted.
cones.group_by("Flavor").len(name="count").sort("Flavor")


There are two distinct categories, chocolate and strawberry. The call to `group_by` creates a DataFrame of counts in each category. The count column is named `count`, and contains the number of rows in each category.

Notice that this can all be worked out from just the `Flavor` column. The `Price` column has not been used.

But what if we wanted the total price of the cones of each different flavor? That's where `sum` comes in.

## Finding a Characteristic of Each Category
After `group_by`, a method such as `sum` aggregates the other columns for all of those rows. For instance, `sum` will sum up the prices in all rows that match each category. This result also contains one row per unique value in the grouped column, but it has the same number of columns as the original DataFrame.

To find the total price of each flavor, we group by `Flavor` again and then call `sum`.

In [ ]:
cones.group_by("Flavor").sum().sort("Flavor")


To create this new DataFrame, `group` has calculated the sum of the `Price` entries in all the rows corresponding to each distinct flavor. The prices in the three `chocolate` rows add up to $\$16.55$ (you can assume that price is being measured in dollars). The prices in the two `strawberry` rows have a total of $\$8.80$.

The summed column keeps the name `Price`. 

Because `sum` reduces every remaining column, and `Price` is the only other column, there is no need to name `Price`.

To see in more detail what `group` is doing, notice that you could have figured out the total prices yourself, not only by mental arithmetic but also using code. For example, to find the total price of all the chocolate cones, you could start by creating a new DataFrame consisting of only the chocolate cones, and then accessing the column of prices:

In [ ]:
cones.filter(pl.col("Flavor") == "chocolate").get_column("Price")


In [ ]:
sum(cones.filter(pl.col("Flavor") == "chocolate").get_column("Price"))


This is what `group` is doing for each distinct value in `Flavor`.

In [ ]:
# For each distinct value in Flavor, access all the rows
# and collect the Price values

cones_choc = cones.filter(pl.col("Flavor") == "chocolate").get_column("Price")
cones_strawb = cones.filter(pl.col("Flavor") == "strawberry").get_column("Price")

# Display the Series in a DataFrame. A column of several Series is stored as lists.

grouped_cones = pl.DataFrame([
    pl.Series("Flavor", ["chocolate", "strawberry"]),
    pl.Series("Array of All the Prices", [cones_choc.to_list(), cones_strawb.to_list()]),
])

# Append a column with the sum of the Price values in each Series

price_totals = grouped_cones.with_columns(
    pl.Series("Sum of the Array", [sum(cones_choc), sum(cones_strawb)])
)
price_totals


You can replace `sum` by another aggregation. For example, `max` finds the largest price in each category:

In [ ]:
cones.group_by("Flavor").max().sort("Flavor")


Once again, `group_by` collects the prices in each `Flavor` category. But now it finds the `max` of each Series:

In [ ]:
price_maxes = grouped_cones.with_columns(
    pl.Series("Max of the Array", [max(cones_choc), max(cones_strawb)])
)
price_maxes


Indeed, the original call to `group` with just one argument has the same effect as using `len` as the function and then cleaning up the DataFrame.

In [ ]:
lengths = grouped_cones.with_columns(
    pl.Series("Length of the Array", [len(cones_choc), len(cones_strawb)])
)
lengths


## Example: NBA Salaries
The DataFrame `nba` contains data on the 2015-2016 players in the National Basketball Association. We have examined these data earlier. Recall that salaries are measured in millions of dollars.

In [ ]:
nba1 = pl.read_csv(path_data + "nba_salaries.csv")
nba = nba1.rename({"'15-'16 SALARY": "SALARY"})
nba


**1.** How much money did each team pay for its players' salaries?

The only columns involved are `TEAM` and `SALARY`. We have to `group` the rows by `TEAM` and then `sum` the salaries of the groups. 

In [ ]:
teams_and_money = nba.select("TEAM", "SALARY")
teams_and_money.group_by("TEAM").sum().sort("TEAM")


**2.** How many NBA players were there in each of the five positions?

We have to classify by `POSITION`, and count. This can be done with just one argument to group:

In [ ]:
nba.group_by("POSITION").len(name="count").sort("POSITION")


**3.** What was the average salary of the players at each of the five positions?

This time, we have to group by `POSITION` and take the mean of the salaries. For clarity, we will work with a DataFrame of just the positions and the salaries.

In [ ]:
positions_and_money = nba.select("POSITION", "SALARY")
positions_and_money.group_by("POSITION").mean().sort("POSITION")


Center was the most highly paid position, at an average of over 6 million dollars.

`.mean()` cannot run on a string column such as `PLAYER` or `TEAM`. The call below uses only `POSITION` and `SALARY`.

In [ ]:
# POLARS_CHANGE: mean errors on string columns, so only POSITION and SALARY are used
nba.select("POSITION", "SALARY").group_by("POSITION").mean().sort("POSITION")
